<a href="https://colab.research.google.com/github/quangminhho004-blip/UWB_RADAR/blob/main/notebooks/DATA_PREPARE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA_PREPARE — chuẩn bị dữ liệu

Chạy **một lần duy nhất**. Khoảng 30 phút.

```
Zenodo 5.7 GB  →  giải nén 13 GB CSV  →  5 script  →  cất ~2.7 GB lên Drive
```

Sau đó mọi notebook thí nghiệm chỉ giải nén từ Drive, mất 2 phút.

**Không cần GPU.** Runtime → CPU cũng chạy được, đỡ tốn quota.

| | | |
|---|---|---|
| repo | `quangminhho004-blip/UWB_RADAR` | code của đồ án |
| upstream | `nesl/mobivital-public` | clone riêng, họ không có LICENSE |
| dataset | Zenodo `10.5281/zenodo.15022885` | `tripod.zip` 5.7 GB |

## 0. Kiểm tra môi trường

Cần ít nhất **20 GB trống**: zip 5.7 GB + CSV giải nén 13 GB.

In [1]:
import os
import subprocess


def chay(lenh):
    """Chạy một lệnh shell, trả về những gì nó in ra."""
    ket_qua = subprocess.run(lenh, shell=True, capture_output=True, text=True)
    return (ket_qua.stdout + ket_qua.stderr).strip()


print(chay("df -h /content | tail -1"))

overlay         236G   48G  189G  21% /


## 1. Mount Google Drive

Dữ liệu đã xử lý sẽ cất ở đây để dùng lại cho mọi thí nghiệm sau.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/mobivital"
os.makedirs(DRIVE, exist_ok=True)
print("sẽ cất kết quả vào", DRIVE)

Mounted at /content/drive
sẽ cất kết quả vào /content/drive/MyDrive/mobivital


## 2. Lấy code

Clone repo đồ án và upstream MobiVital. `external/mobivital/` bị `.gitignore` chặn
nên phải clone riêng mỗi phiên.

In [3]:
REPO = "/content/UWB_RADAR"

# Kiểm tra ".git" chứ không chỉ kiểm tra thư mục: có thư mục mà thiếu .git
# thì không biết đang chạy bản code nào.
if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    print(chay("git pull -q origin main && echo 'đã cập nhật code mới nhất'"))
else:
    os.chdir("/content")          # bước ra ngoài trước, không thì xoá mất chỗ đang đứng
    chay("rm -rf " + REPO)
    print(chay("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO))
    print(chay("git clone -q https://github.com/nesl/mobivital-public.git " + REPO + "/external/mobivital"))
    print(chay("pip install -q einops"))

os.chdir(REPO)
print("đang đứng ở:", os.getcwd())
print(chay("ls"))
print()
print("commit code của mình :", chay("git rev-parse --short HEAD"))
print("commit MobiVital     :", chay("git -C external/mobivital rev-parse --short HEAD"))

đang đứng ở: /content/UWB_RADAR
docs
external
notebooks
README.md
results
scripts
src
commit code của mình : 82952a1
commit MobiVital     : 4319731


## 3. Tải dataset từ Zenodo

Dùng `aria2c` chia 16 luồng, **không dùng `wget`**. Đo thật trên Colab:

| | tốc độ | thời gian cho 5.7 GB |
|---|---|---|
| `wget` | 0.7 MB/s | **2.3 giờ** |
| `aria2c -x16` | 56 MB/s | **2 phút** |

Zenodo bóp băng thông mỗi kết nối, nên chia nhiều luồng ăn ngay.

In [4]:
ZIP = "/content/tripod.zip"
KICH_THUOC_DUNG = 5700705593      # byte, lấy từ Zenodo API

if os.path.exists(ZIP) and os.path.getsize(ZIP) == KICH_THUOC_DUNG:
    print("đã có sẵn, bỏ qua bước tải")
else:
    chay("apt-get install -qq -y aria2")
    # --summary-interval=0 tắt bảng tiến trình, không thì in ra 250 dòng
    # làm phình notebook. Chỉ giữ dòng kết quả cuối.
    ket_qua = chay("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
                   "-d /content -o tripod.zip "
                   "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    for dong in ket_qua.split("\n"):
        if "OK" in dong or "ERR" in dong:
            print(dong)

kich_thuoc = os.path.getsize(ZIP)
print("%s  %.2f GB" % (ZIP, kich_thuoc / 1e9))
print("đúng kích thước Zenodo công bố:", kich_thuoc == KICH_THUOC_DUNG)

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
fcca01|OK  |    24MiB/s|/content/tripod.zip
Status Legend:
(OK):download completed.
/content/tripod.zip  5.70 GB
đúng kích thước Zenodo công bố: True
[#fcca01 5.3GiB/5.3GiB(99%) CN:1 DL:11MiB]


## 4. Giải nén

5.7 GB nén thành **13 GB CSV**, 1874 file. Mất khoảng 3 phút.

Mỗi file là một buổi ghi 30 giây: **1500 dòng × 254 cột**, lấy mẫu 50 Hz.

In [5]:
# Trong zip đã có sẵn thư mục "tripod/", nên giải nén thẳng vào data/raw/
# chứ KHÔNG vào data/raw/tripod/ — không thì lồng hai tầng
# data/raw/tripod/tripod/*.csv và script 1 tìm không ra.
os.makedirs("data/raw", exist_ok=True)
chay("unzip -q -o " + ZIP + " -d data/raw/")

print("số file CSV:", chay("find data/raw -name '*.csv' | wc -l"))
print("dung lượng :", chay("du -sh data/raw | cut -f1"))
print()
print("cấu trúc:", os.listdir("data/raw"))
print("file mẫu :", sorted(os.listdir("data/raw/tripod"))[0])

số file CSV: 1874
dung lượng : 13G
cấu trúc: ['tripod']
file mẫu : 231012_userI_tripod_01_0.csv


## 5. Gom CSV theo người

`data/raw/tripod/` 1874 file → `data/raw/A/` … `data/raw/L/`.

Tên file dạng `240409_userG_tripod_02_3.csv`, lấy chữ cái ngay sau `user`.

```
phát triển  A B C D E F K L   1289 buổi ghi
để dành     G H I J            537 buổi ghi
```

Chia theo người, không chia theo buổi ghi — `docs/WHY_SPLIT_BY_USER.md`.

In [6]:
print(chay("python scripts/prepare_raw.py"))
print()
print(chay("for u in A B C D E F G H I J K L; do "
           "printf '%s %4d  ' $u $(ls data/raw/$u | wc -l); done"))

Tìm thấy 1874 file CSV
Đã chuyển 1874 file
A  225  B  159  C  214  D  208  E  128  F  110  G  136  H  143  I  152  J  128  K  152  L  119


## 6. → `by_user/*.npz` — pipeline DEV

Mỗi người một file, ba mảng:

```
uwb    (n, 1500, 120) complex64   radar, 120 khoảng cách
gt     (n, 1500)      float32     nhịp thở thật, chuẩn hoá [-1, 1]
files  (n,)                       tên file CSV
```

Bố cục CSV: cột `12–131` phần thực, `132–251` phần ảo, cột áp chót nhịp thở thật.
Bỏ file không đủ 1500 dòng — có 48 file.

Đọc bằng `csv.reader` rồi ép `float32` ngay, giống MobiVital. Đọc `float64` rồi mới
ép xuống thì lệch ở chữ số thứ 8, bước 8 sẽ không ra 0.

In [7]:
print(chay("python scripts/make_npz.py"))

user A -> data/processed/by_user/A.npz | uwb (224, 1500, 120) | gt (224, 1500) | files (224,)
user B -> data/processed/by_user/B.npz | uwb (156, 1500, 120) | gt (156, 1500) | files (156,)
user C -> data/processed/by_user/C.npz | uwb (211, 1500, 120) | gt (211, 1500) | files (211,)
user D -> data/processed/by_user/D.npz | uwb (206, 1500, 120) | gt (206, 1500) | files (206,)
user E -> data/processed/by_user/E.npz | uwb (126, 1500, 120) | gt (126, 1500) | files (126,)
user F -> data/processed/by_user/F.npz | uwb (102, 1500, 120) | gt (102, 1500) | files (102,)
user G -> data/processed/by_user/G.npz | uwb (134, 1500, 120) | gt (134, 1500) | files (134,)
user H -> data/processed/by_user/H.npz | uwb (138, 1500, 120) | gt (138, 1500) | files (138,)
user I -> data/processed/by_user/I.npz | uwb (145, 1500, 120) | gt (145, 1500) | files (145,)
user J -> data/processed/by_user/J.npz | uwb (120, 1500, 120) | gt (120, 1500) | files (120,)
user K -> data/processed/by_user/K.npz | uwb (148, 1500, 120

## 7. → `external/mobivital/data_final/*.npy` — pipeline GỐC

Chạy `prep_breath_final.py` của MobiVital, 0 dòng sửa, bằng đúng lệnh trong
README của họ.

Code họ dùng đường dẫn tương đối `./dataset/mobivital/tripod/`, nên
`scripts/mobivital/setup_dataset.py` dựng thư mục đó ngay trong repo họ bằng
symlink rồi ghi vào `.git/info/exclude` — loại trừ cục bộ, không thuộc repo.
`data_final/` thì chính script của họ sinh ra.

Hai file `.npy` sinh ra dùng ở bước 8 và ở `windows/final_train`.


In [ ]:
print(chay("python scripts/mobivital/setup_dataset.py")[-600:])
print()
print(chay("cd external/mobivital && python dataset_preparation/prep_breath_final.py 2>&1 | tail -2"))
print(chay("ls -la external/mobivital/data_final/"))
print()
print("KIỂM TRA: không được sửa gì trong repo MobiVital")
print(chay("git -C external/mobivital status --short") or "  git diff trống — không sửa dòng nào")


## 8. Đối chiếu hai pipeline

Phải in ra:

```
Khac nhau nhieu nhat tren 1500 mau: 0.0
```

Hai pipeline đọc cùng bộ CSV bằng hai đường khác nhau. Sai lệch 0 nghĩa là
`by_user/*.npz` chứa đúng những con số mà `data_final/*.npy` chứa.

Từ đó các thí nghiệm sau chỉ đọc `by_user/*.npz`, bỏ được CSV thô 13 GB và
`data_final/` 2.6 GB.

Không ra 0 thì dừng.


In [9]:
print(chay("python scripts/check_data.py"))

PHAN 1 — 12 file .npz cua minh
nguoi A | 224 session | uwb (224, 1500, 120) | gt (224, 1500) | gt tu -1.0 den 1.0
nguoi B | 156 session | uwb (156, 1500, 120) | gt (156, 1500) | gt tu -1.0 den 1.0
nguoi C | 211 session | uwb (211, 1500, 120) | gt (211, 1500) | gt tu -1.0 den 1.0
nguoi D | 206 session | uwb (206, 1500, 120) | gt (206, 1500) | gt tu -1.0 den 1.0
nguoi E | 126 session | uwb (126, 1500, 120) | gt (126, 1500) | gt tu -1.0 den 1.0
nguoi F | 102 session | uwb (102, 1500, 120) | gt (102, 1500) | gt tu -1.0 den 1.0
nguoi K | 148 session | uwb (148, 1500, 120) | gt (148, 1500) | gt tu -1.0 den 1.0
nguoi L | 116 session | uwb (116, 1500, 120) | gt (116, 1500) | gt tu -1.0 den 1.0
nguoi G | 134 session | uwb (134, 1500, 120) | gt (134, 1500) | gt tu -1.0 den 1.0
nguoi H | 138 session | uwb (138, 1500, 120) | gt (138, 1500) | gt tu -1.0 den 1.0
nguoi I | 145 session | uwb (145, 1500, 120) | gt (145, 1500) | gt tu -1.0 den 1.0
nguoi J | 120 session | uwb (120, 1500, 120) | gt (120, 

## 9. Cắt cửa sổ để train

```
200 mẫu vào → 25 mẫu phải đoán, trượt 25  ⟹  52 cửa sổ mỗi sóng
```

Gọi `generate_dataset` của MobiVital, cắt riêng từng người để ghép fold tuỳ ý:

```
fold 1  train C+D+E+F+K+L   val A+B
fold 2  train A+B+D+F+K+L   val C+E
fold 3  train A+B+C+E+K+L   val D+F
fold 4  train A+B+C+D+E+F   val K+L
```

Cắt riêng 8 người rồi ghép cho ra đúng tập cửa sổ như cắt một lần — 292.708 cửa sổ
cả hai cách, chỉ khác thứ tự.

Cửa sổ chỉ dùng để **train**. Chấm điểm đọc buổi ghi thô từ `by_user/*.npz`, vì
việc chọn sóng ở bước này có nhìn nhịp thở thật (`corr > 0.9`).

In [10]:
print(chay("python scripts/make_windows.py"))

PHẦN 1 — pipeline DEV, cắt riêng từng người
ngưỡng 0.9
data/processed/windows/dev_cv/A_corr0.9_h200_f25.npz | X (42640, 200) | 37 MB
mất 3 giây
data/processed/windows/dev_cv/B_corr0.9_h200_f25.npz | X (39104, 200) | 34 MB
mất 2 giây
data/processed/windows/dev_cv/C_corr0.9_h200_f25.npz | X (47996, 200) | 41 MB
data/processed/windows/dev_cv/D_corr0.9_h200_f25.npz | X (53300, 200) | 46 MB
data/processed/windows/dev_cv/E_corr0.9_h200_f25.npz | X (25792, 200) | 22 MB
data/processed/windows/dev_cv/F_corr0.9_h200_f25.npz | X (9256, 200) | 8 MB
mất 1 giây
data/processed/windows/dev_cv/K_corr0.9_h200_f25.npz | X (50804, 200) | 44 MB
data/processed/windows/dev_cv/L_corr0.9_h200_f25.npz | X (23816, 200) | 20 MB
PHẦN 2 — pipeline GỐC, cắt gộp 8 người
đọc external/mobivital/data_final/training_breath_tripod_data.npy -> 1289 session
data/processed/windows/final_train/train_corr0.9_h200_f25.npz | X (292708, 200) | 251 MB
mất 10 giây
Xong. Kết quả ở data/processed/windows


## 10. Cất lên Drive

| | dùng để | dung lượng |
|---|---|---|
| `windows_dev_cv.tar.gz` | train | ~250 MB |
| `by_user.tar` | chấm điểm | ~2.4 GB |

Không đưa lên:

- `data/raw/` 13 GB — tải lại từ Zenodo mất 2 phút
- `external/mobivital/data_final/` 2.6 GB — sinh lại bằng script của họ
- `windows/final_train/` 251 MB — là 8 file `dev_cv` ghép lại

`by_user` không nén: gzip chỉ giảm 6% (147 MB → 138 MB) mà tốn thêm vài phút.


In [13]:
chay("tar -czf " + DRIVE + "/windows_dev_cv.tar.gz -C data/processed/windows dev_cv")
chay("tar -cf  " + DRIVE + "/by_user.tar          -C data/processed by_user")

print(chay("ls -la " + DRIVE))
print()
print("Drive đang dùng:", chay("du -sh " + DRIVE + " | cut -f1"))

total 2630692
-rw------- 1 root root 2640629760 Sep  3 08:28 by_user.tar
-rw------- 1 root root   53197934 Sep  3 08:25 windows_dev_cv.tar.gz
Drive đang dùng: 2.6G


## Xong

Từ giờ mọi notebook thí nghiệm bắt đầu bằng ô này, mất khoảng 2 phút:

```python
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!git clone -q https://github.com/nesl/mobivital-public.git external/mobivital
!pip install -q einops

!mkdir -p data/processed
!tar -xzf /content/drive/MyDrive/mobivital/windows_dev_cv.tar.gz -C data/processed/
!tar -xf  /content/drive/MyDrive/mobivital/by_user.tar          -C data/processed/

!ln -s /content/drive/MyDrive/mobivital/runs runs
```

`runs/` là **lối tắt trỏ vào Drive** — Colab hay ngắt phiên giữa chừng, checkpoint
ghi vào đó thì phiên sau `resume` chạy tiếp được.